# Ingestion Pipeline

End-to-end pipeline that:
1. Loads `.txt` knowledge-base files
2. Parses metadata from file headers
3. Chunks the documents
4. Embeds the chunks
5. Stores them in the vector database
6. Runs a sanity-check retrieval query

**Reuses** logic from `1_load_chunk.ipynb`, `2_embeddings.ipynb`, `3_retrieval.ipynb`, `4_basic_RAG.ipynb` and `src/` py files.

---
## Setup 
### Imports & Paths

In [ ]:
from pathlib import Path
import sys

# ── Project root & src on path ──────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# ── Knowledge-base directories ───────────────────────────────────────────────
LGBT_EU_BY_COUNTRY_DIR = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_country"
LGBT_EU_BY_SUBSET_DIR  = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_subset"
HIV_KB_DIR             = PROJECT_ROOT / "data" / "3_txt_KB" / "HIV_AIDS_data"
UNICEF_KB_DIR          = PROJECT_ROOT / "data" / "3_txt_KB" / "UNICEF_Immunization"

KB_DIRS = [
    LGBT_EU_BY_COUNTRY_DIR,
    LGBT_EU_BY_SUBSET_DIR,
    HIV_KB_DIR,
    UNICEF_KB_DIR,
]

print("Project root:", PROJECT_ROOT)
for d in KB_DIRS:
    status = "✓" if d.exists() else "✗ (not found)"
    print(f"  {status}  {d.relative_to(PROJECT_ROOT)}")

In [ ]:
# ── Standard library & third-party ───────────────────────────────────────────
from typing import Dict, List, Tuple

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# ── src modules (reused from previous notebooks) ─────────────────────────────
from vectorstore import load_vectorstore
from retrieval   import retrieve, print_results, format_context
from llm         import build_prompt, ask

print("All imports OK.")

---
## Parameters

- uses `RecursiveCharacterTextSplitter` used in `1_load_chunk.ipynb`.
- Edit params to change how it will be applied.

In [ ]:
# ── Chunking parameters (mirrors 1_load_chunk.ipynb) ─────────────────────────
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

# ── Vector-store persistence path ─────────────────────────────────────────────
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore"

# ── Embedding model (mirrors 2_embeddings.ipynb) ──────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"

# ── Retrieval parameters (mirrors 3_retrieval.ipynb) ─────────────────────────
TOP_K = 5

print(f"CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}")
print(f"EMBEDDING_MODEL={EMBEDDING_MODEL}")
print(f"VECTORSTORE_DIR={VECTORSTORE_DIR}")

---
## Load All `.txt` Files
- load in from knowledge base (the output of `7_generate_text_knowledge_base.ipynb`)